In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import Code
from geopy.geocoders import Nominatim

In [ ]:
output_dir = Path('outputs/dataset_preparation')
if not output_dir.exists():
  output_dir.mkdir(parents=True)

data_dir = Path('data/dataset')
if not data_dir.exists():
  data_dir.mkdir(parents=True)

download_toml_dir = Path('../configs/download')

In [ ]:
from collections import namedtuple

# The zoom-ins will be around the following area.

# Location = namedtuple('Location', ('latitude', 'longitude'))
# location = Location(latitude=-89.9277, longitude=163.25)
geolocator = Nominatim(user_agent='dataset_preparation')
location = geolocator.geocode('Greenwich')
width, height = 20, 5
bounds = [location.longitude - width / 2, location.longitude + width / 2,
          location.latitude - height / 2, location.latitude + height / 2]

In [ ]:
def _wrap(da: xr.DataArray, longitudes: np.ndarray):
    """ Wraps around longitude dimension. It assumes that longitude is the last dimension."""
    size = longitudes.size
    dayline_index = np.argmax(longitudes > 180.0)
    data = da.data
    wrapped = da.copy()
    wrapped.data[..., :(size - dayline_index)] = data[..., dayline_index:]
    wrapped.data[..., (size - dayline_index):] = data[..., :dayline_index]
    return wrapped

def fix_longitude_convention(da: xr.DataArray):
    longitude = da.coords['longitude'].copy()
    da_wrapped = _wrap(da, longitude.to_numpy())
    lon_wrapped = _wrap(longitude, longitude.to_numpy())
    lon_wrapped = lon_wrapped.to_numpy()
    lon_wrapped = np.where(lon_wrapped <= 180.0, lon_wrapped, lon_wrapped - 360.0)
    lon_wrapped = xr.DataArray(data=lon_wrapped, coords={'longitude': lon_wrapped}, dims='longitude', name='longitude')
    da_wrapped = da_wrapped.assign_coords(longitude=lon_wrapped)
    return da_wrapped

In [ ]:
def plot(da, dpi=100, land=True, colorbar=False, zoomin=False, **kwargs):
    fig = plt.figure(dpi=dpi)
    # Notice: some projections will display incorrectly when using the ECMWF convention 
    # for longitudes (i.e., between 0 and 360 excluded) is used, others like
    # InterruptedGoodeHomolosine(emphasis='ocean') are broken. 
    # Therefore, it is advisable to manually wrap the data around longitude dimension.
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.InterruptedGoodeHomolosine(emphasis='ocean'))
    if zoomin:
        ax.set_extent(bounds)
    da = fix_longitude_convention(da)
    lons, lats = np.meshgrid(da.longitude, da.latitude)
    if land:
        ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='black', alpha=0.5))
    img = ax.imshow(da, transform=ccrs.PlateCarree(), **kwargs)
    if colorbar:
        fig.colorbar(img, orientation='horizontal')
    plt.show()

# Technical comments on dataset preparation

The dataset used to train the model is derived from GLORYS12 and ERA5. Since the original datasets are huge, most of the post-processing is performed online while downloading, leveraging the lazy loading feature of some backends. The implementation of the download and post-processing routines is in the `dataset.py` module, which exposes a CLI and provides a common interface to download files from Climate Data Store, Copernicus Marine Data Store, and Zarr buckets on Google Cloud Storage (GCS).

The user has to provide a TOML file containing pieces of information about the provider (`cm` for `copernicusmarine`, `cds` for `cdsapi`, and `gcs` for Zarr stored on GCS), the dataset (e.g., its name and variables to download), whether to dump on disk intermediate results to limit memory consumption, whether to download only a subsequence to limit execution time (when invoked in a SLURM array), post-processing, and finally saving. Finally, the output path has to be specified via command line argument.

As noted, `dataset.py` implements several hacks to stay within the memory constraints currently imposed on Leonardo nodes connected to the internet. In particular, the range of dates to be downloaded is divided in subsequences, which are then processed and saved one at a time on disk (a local SSD, the temporary location is managed by Python `tempfile`). Notice that when using the `cds` provider several other temporary files are created, due to the design of `cdsapi`. Finally, the temporary Zarr dataset is optionally re-chunked and saved to the output path.


## The TOML configuration file

A single TOML configuration file can handle multiple datasets. This originates from design choices related to the Copernicus Marine Service. As an example, the biogeochemical reanalysis of the Mediterranean Sea splits its variables into several datasets or products. This way one can download, post-process, and output in a single Zarr dataset all variables.

An example of a TOML configuration file is the following:

In [ ]:
Code(download_toml_dir / 'glorys12_bathymetry_tres-static_res-0.25_levels-10.toml')

The order of post-processing steps is hardcoded and defined in the `dataset_utils.py`. Notice that the order matters, both in terms of results and performance. However, the system is quite extensible. All available `xarray.Dataset` methods can be added just by modifying the `PostprocessingSteps` literal type, and custom one by adding methods to the `Postprocess` class. The current strategy is to resample, then chunk, and then regrid.

## About Regridding

One of the post-processing steps is `regrid`, and requires particular care (see [here](https://climatedataguide.ucar.edu/climate-tools/regridding-overview)).

Regridding is provided by the `xarray-regrid` Python package, but an `interpolate` method based on `xarray` (see the documentation for available algorithms) is also supplied. However, traditional interpolation routines do not consider the spherical geometry of the earth. Hence, their usage is discouraged.

In most applications, conservative regridding (i.e. preserving the integral of the source field across the regridding) is the go-to choice for continuous scalar fields. Notice that both GraphCast and WeatherBench datasets use first-order conservative regridding also for vector fields. The implementation used by `xarray-regrid` is based on the `conservative_normed` method provided by `xESMF`, plus minor [modifications suggested by Stephan Hoyer](https://github.com/xarray-contrib/xarray-regrid/blob/eef312a2ee0fa5cae94814d72f17438f77ec9e5b/src/xarray_regrid/methods/conservative.py#L49-L54).

Finally, notice xarray-regrid takes [different longitude conventions into account](https://github.com/xarray-contrib/xarray-regrid/blob/eef312a2ee0fa5cae94814d72f17438f77ec9e5b/src/xarray_regrid/utils.py#L341-L390).
This is useful as [ERA5 uses a different convention for longitude](https://confluence.ecmwf.int/display/CKB/ERA5%3A+What+is+the+spatial+reference#ERA5:Whatisthespatialreference-Coordinatesystem).

### Note on GLORYS12 and its bathymetry

The `conservative_normed` algorithm relies on computing overlapping areas between cells of the source and destination grid, then normalizing for the fraction of source grid cells actually containing a value. This allows handling missing values (NaNs) without adding too much overhead. It is important when regridding GLORYS12, as the domain covered by this dataset contains land points. This also is the reason why Dask will raise a `RuntimeWarning: invalid value encountered in divide` during execution: a destination grid cell not overlapping with source grid cells containing water points (e.g. within land masses) entails division by zero.

Another crucial point is how to regrid the bathymetry. The nearest neighbor interpolation is usually the [right choice for categorical variables](https://climatedataguide.ucar.edu/climate-tools/regridding-overview): in this case, 0 is land, and 1 is sea. Using other interpolation algorithms would mean to work with meaningless values, or the choice of some arbitrary threshold. Together with previous considerations, they imply that there will be points in the regridded dataset with non-null value marked as land around coastlines. This is usually not a problem, as they can be masked afterward.

Notice that `xarray-regrid` will convert everything to floats, and after regridding the bathymetry will be contained in a `xarray.DataArray` containing single precision floats. This is the reason why it is necessary to unsafely re-cast them to `int8` in a following post-processing step (and why Dask will complain with a `RuntimeWarning: invalid value encountered in cast return x.astype(astype_dtype, **kwargs)`).

In [ ]:
# Download mask from Copernicus Marine using provided download script and configuration file
bathymetry_path = data_dir / 'glorys12_bathymetry_tres-static_res-0.25_levels-10.zip'
bathymetry = xr.open_zarr(bathymetry_path)
mask = bathymetry['mask'].isel(depth=0, drop=True)
mask = mask.compute()
mask

In [ ]:
# Download GLORYS12 sample (only for surface variables) from Copernicus Marine 
# using provided download script and configuration file
glorys12_path = data_dir / 'glorys12_phys_tres-1d_res-0.25_levels-10.zip'
glorys12 = xr.open_zarr(glorys12_path)
sst = glorys12['thetao'].isel(depth=0, time=0, drop=True)
sst = sst.compute()
sst

In [ ]:
# Download ERA5 sample (only for surface variables) from WeatherBench Google Cloud Storage bucket 
# using provided download script and configuration file
era5_path = data_dir / ('era5_tres-1d_res-0.25_levels-1.zip')
era5 = xr.open_zarr(era5_path)
sat = era5['2m_temperature'].isel(time=1, drop=True)
sat = sat.compute()
sat

In [ ]:
# Download ERA5 sample (only for surface variables) from WeatherBench Google Cloud Storage bucket 
# using provided download script and configuration file
prec = era5['total_precipitation'].isel(time=1)
prec = prec.compute()
prec

In [ ]:
plot(mask, origin='lower')

In [ ]:
plot(sst, origin='lower', colorbar=True)

In [ ]:
plot(sst.notnull() & np.logical_not(mask), origin='lower', land=False)

In [ ]:
plot(sst.notnull() & np.logical_not(mask), zoomin=True)

In [ ]:
plot(sst.where(mask, np.nan), zoomin=True, colorbar=True)

In [ ]:
plot(glorys12['thetao'].isel(depth=4, time=0, drop=True))

In [ ]:
plot(glorys12['siconc'].isel(time=0, drop=True), colorbar=True)

### Note on ERA5

The GLORYS12 reanalysis is forced using ERA5, hence it seems natural to use the same dataset for atmospheric forcing of a data-driven model trained on the previous. Moreover, ERA5 is widely used in both numerical and machine learning communities, and can be downloaded from several sources.

The reference is the [Climate Data Store](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview), which can be accessed using a Python API provided by the `cdsapi` package, and supported by the downloader developed and used for this project. It contains the ERA5 reanalysis regridded on a 0.25-degree grid, which is the same target grid used for GLORYS12. However, it is usually slower to go that way, at very least because it requires downloading several intermediate files.

Another option is using [WeatherBench datasets GCS bucket](https://weatherbench2.readthedocs.io/en/latest/data-guide.html), which contains already postprocessed versions of the dataset available from the Climate Data Store. These analysis-ready-cloud-optimized datasets are usually faster to download (WeatherBench GCS provides the dataset with chunk sizes `{time = 1, level = 13, lat = 721, lon = 1440}` amounting to about 54 MB per chunk per variable). However, they do not contain all variables available in ERA5. This is not a limitation if one is developing an ocean model coupled with existing weather models (e.g. GraphCast).

Also, notice that when working at 1-degree resolution, one might use an older version of the ERA5 dataset, provided by WeatherBench for continuity, and avoid regridding: `gs://weatherbench2/datasets/era5/1959-2022-1h-360x181_equiangular_with_poles_conservative.zarr`. In that case, precipitation should be treated separately, as it is the only variable that requires accumulation during resampling. Also, the dataset occasionally contains negative values for total precipitation, which have to be set to zero.

The dataset at `gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr` already contains the precipitation accumulated over a 24-hour period among other derived variables (`total_precipitation_24hr`). As for the documentation, negative values should have been taken care of. However, it appears not.

The question of which atmospheric variables one should use to train a data-driven global ocean circulation model is still an open research one. An obvious starting point is using the variables used by numerical models, as NEMO or MITgcm. The list of those used by the latter could be found in the [MITgcm documentation](https://mitgcm.readthedocs.io/en/latest/phys_pkgs/exf.html#ssub-phys-pkg-exf-inputs-units). These variables are used in so called bulk parametrizations, for a discussion of those used in NEMO one can read the following [paper](https://doi.org/10.5194/gmd-15-6873-2022). For example, the stresses, evaporation, and latent and sensible heat fluxes can be derived using bulk formulae from the air density among other quantities, which can be derived itself from temperature, specific humidity, and pressure. In particular, we included the dew point temperature but not the lowest level of the 3D specific humidity, following the [ECMWF guidelines for computing the surface (2m) specific humidity](https://confluence.ecmwf.int/display/CKB/ERA5%3A+data+documentation#heading-Guidelines). Therefore, we selected only the variables listed in the `era5_tres-1d_res-0.25_levels-1.toml` file. Notice that one can use the `mean` reduction method for resampling of all variables, both instantaneous (like 10-meter wind speed), and accumulated (like precipitation and runoff), as the latter is just a rescaling of the accumulated value.

We also included variables related to radiation, contrary to other data-driven ocean models. Indeed, the 2-meter temperature is strongly correlated with the sea surface temperature, making the other variables redundant if the former is given at each timestep. However, we decided to include all relevant variables with the hope that the model would learn physical relationships between them, and in perspective of a future coupling with a weather model.

An ARCO version of ERA5 is available at `gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3`, and the accompanying [GitHub repository](https://github.com/google-research/arco-era5) is a good reference on how to prepare and store large Earth system datasets.

### Note on Chunking

Chunk size is a tradeoff between Zarr data format (good chunking practice for cloud storage seems to be around 1-10MB), and Lustre file system, which would work best with large files (around 1GB).
With the selected resolution, the chunk size is about 4MB (float32) per variable per day without compression (zarr use lz4 compression by default with level 5 via blosc, resulting in about half the size).
With dask auto chunking, the target chunk size should be 128 MiB by default, see: https://docs.dask.org/en/latest/array-chunks.html#automatic-chunking

Currently, we're adopting a multi-file dataset consisting of zipped Zarrs with target size of about 10GB.

In [ ]:
Code(download_toml_dir / 'era5_tres-1d_res-0.25_levels-1.toml')


0. load bathymetry, GLORYS12, and ERA5 dataset
1. Re-center ERA5 dataset time (day before, due to mean reduce method)
2. Invert ERA5 latitudes
3. Fill GLORYS sea ice nans with zero (inside mask)
4. Rename depth dim to levels, keep the depth dimension (but not as an index dimension)
5. Rename latitude and longitude as lat and lon, both the dimensions and coordinates
6. Check that each dataset contains all the days between beginning and end
7. Check that each dataset uses the [0, 360) convention for longitude
  8. Check that each dataset contains all latitudes in [-90, 90] (including antarctica), and use the [90, -90] convention (or whatever is used by GraphCast)
9. Check that positive variables do not contain negative values (e.g. precipitations)
10. Check that GLORYS12 dataset does not contain nan values outside mask, print the days of the dataset that contain a nan
11. Check that ERA5 dataset does not contain nans, print the days of the dataset that contain a nan
12. Save the dataset in a ZipStore Zarr, with sensible chunk sizes


In [ ]:
plot(prec, colorbar=True)

In [ ]:
plot(prec < 0.0)

In [ ]:
mask.latitude

In [ ]:
era5.latitude

In [ ]:
era5

In [150]:
ds = xr.load_dataset('../data/dataset/glofas_static_tres-static_res-0p25_levels-1.zip', engine='zarr')
ds

<xarray.Dataset> Size: 17MB
Dimensions:    (lat: 721, lon: 1440)
Coordinates:
  * lat        (lat) float32 3kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * lon        (lon) float32 6kB 0.0 0.25 0.5 0.75 ... 359.0 359.2 359.5 359.8
Data variables:
    elevation  (lat, lon) float64 8MB nan nan nan nan nan ... nan nan nan nan
    uparea     (lat, lon) float64 8MB nan nan nan nan nan ... nan nan nan nan
Attributes:
    CDI:                        Climate Data Interface version 1.9.10 (https:...
    CDO:                        Climate Data Operators version 1.9.10 (https:...
    Conventions:                CF-1.9
    GDAL:                       GDAL 3.0.4, released 2020/01/28
    GDAL_AREA_OR_POINT:         Area
    NCO:                        netCDF Operators version 4.9.2 (Homepage = ht...
    history:                    Tue Nov 15 12:14:24 2022: cdo chname,Band1,el...
    history_of_appended_files:  Fri Nov 20 14:22:39 2020: Appended file /huge...

In [ ]:
ds['